In [1]:
# PARAMETERS CELL
processing_date = "2026-08-28"
run_mode = "full"

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, DateType, BooleanType
from datetime import datetime

from datetime import datetime, timezone
SILVER_RUN_TS = datetime.now(timezone.utc).isoformat()
PROCESSING_DATE = processing_date
print(f"Silver layer run started at {SILVER_RUN_TS} for processing_date={PROCESSING_DATE}")

dq_log_rows = []


def log_dq(table_name: str, total_in: int, total_out: int, quarantined: int, notes: str = ""):
    dq_log_rows.append({
        "run_ts": SILVER_RUN_TS,
        "table_name": table_name,
        "rows_in": total_in,
        "rows_out": total_out,
        "rows_quarantined": quarantined,
        "quarantine_rate_pct": round((quarantined / total_in * 100), 2) if total_in > 0 else 0.0,
        "notes": notes,
    })
    print(f"[DQ] {table_name}: in={total_in} out={total_out} quarantined={quarantined} ({notes})")


print(f"Silver layer run started at {SILVER_RUN_TS}")

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 4, Finished, Available, Finished, False)

Silver layer run started at 2026-09-03T07:48:41.254465+00:00 for processing_date=2026-08-28
Silver layer run started at 2026-09-03T07:48:41.254465+00:00


**Silver: Customers**

Deduplicates exact-content duplicate customer rows (seeded intentionally in
Bronze to simulate a real upstream duplication bug), standardizes email
(lowercase, blank → null) and phone formatting, and casts `customer_id` to
integer.

**Strategy:** duplicates are not dropped — they're quarantined into
`silver_quarantine_customers` for audit, while the canonical row (lowest
`customer_id`) survives into `silver_customers`. Missing emails are kept
and flagged via `has_missing_email` rather than rejected, since a missing
email doesn't invalidate the rest of the customer record.



In [3]:
customers_bronze = spark.table("bronze_customers")

# Standardize email: lowercase, treat blank string as null
customers_std = customers_bronze.withColumn(
    "email_clean",
    F.when((F.trim(F.col("email")) == "") | F.col("email").isNull(), None)
     .otherwise(F.lower(F.trim(F.col("email"))))
).withColumn(
    "phone_clean",
    F.regexp_replace(F.col("phone"), r"[\s\-\(\)]", "")
).withColumn(
    "customer_id", F.col("customer_id").cast(IntegerType())
)

# Deduplication: exact-content duplicates get canonicalized to the lowest customer_id
window_cols = ["first_name", "last_name", "email_clean", "phone_clean", "city", "country"]
from pyspark.sql.window import Window
dedup_window = Window.partitionBy(*window_cols).orderBy(F.col("customer_id").asc())

customers_ranked = customers_std.withColumn("_dedup_rank", F.row_number().over(dedup_window))

silver_customers = (
    customers_ranked.filter(F.col("_dedup_rank") == 1)
    .select(
        "customer_id", "first_name", "last_name",
        F.col("email_clean").alias("email"),
        F.col("phone_clean").alias("phone"),
        "city", "country",
        F.col("_bronze_ingested_at"),
    )
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
    .withColumn("has_missing_email", F.col("email").isNull())
)

quarantine_customers = customers_ranked.filter(F.col("_dedup_rank") > 1)

silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
quarantine_customers.write.format("delta").mode("overwrite").saveAsTable("silver_quarantine_customers")

log_dq(
    "silver_customers",
    total_in=customers_bronze.count(),
    total_out=silver_customers.count(),
    quarantined=quarantine_customers.count(),
    notes="exact-duplicate rows removed, canonical = lowest customer_id",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 5, Finished, Available, Finished, False)

[DQ] silver_customers: in=520 out=510 quarantined=10 (exact-duplicate rows removed, canonical = lowest customer_id)


**Silver: Products**

Type casts only — `product_id` to integer, `unit_price` to double,
`is_active` to boolean. No data quality issues were seeded in this source,
so no quarantine logic is needed here.


In [4]:
products_bronze = spark.table("bronze_products")

silver_products = (
    products_bronze
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
    .withColumn("is_active", F.col("is_active").cast(BooleanType()))
    .select("product_id", "product_name", "category", "unit_price", "is_active", "_bronze_ingested_at")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
)

silver_products.write.format("delta").mode("overwrite").saveAsTable("silver_products")

log_dq(
    "silver_products",
    total_in=products_bronze.count(),
    total_out=silver_products.count(),
    quarantined=0,
    notes="type casts only, no quality issues seeded in this source",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 6, Finished, Available, Finished, False)

[DQ] silver_products: in=100 out=100 quarantined=0 (type casts only, no quality issues seeded in this source)


**_Silver: Orders_**

Type casts `order_id` and `customer_id` to integer, and parses
`order_date` into a proper timestamp. No quality issues were seeded here
either — this table exists mainly to support the fact table join in Gold.

In [5]:
orders_bronze = spark.table("bronze_orders")

silver_orders = (
    orders_bronze
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("order_date", F.to_timestamp(F.col("order_date")))
    .select("order_id", "customer_id", "order_date", "order_status", "_bronze_ingested_at")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
)

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

log_dq(
    "silver_orders",
    total_in=orders_bronze.count(),
    total_out=silver_orders.count(),
    quarantined=0,
    notes="type casts only",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 7, Finished, Available, Finished, False)

[DQ] silver_orders: in=2003 out=2003 quarantined=0 (type casts only)


**Silver: Order Items**

Type casts numeric fields, then performs a **referential integrity check**
against `silver_products`: any `order_item` whose `product_id` doesn't
exist in the products table gets quarantined rather than dropped silently.

This simulates a real upstream problem — a line item referencing a product
that was deleted, mistyped, or never synced correctly. Quarantining
preserves the evidence so the business can investigate, instead of just
losing the row.

In [6]:
order_items_bronze = spark.table("bronze_order_items")
valid_product_ids = spark.table("silver_products").select("product_id")

order_items_typed = (
    order_items_bronze
    .withColumn("order_item_id", F.col("order_item_id").cast(IntegerType()))
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
)

order_items_checked = order_items_typed.join(
    valid_product_ids.withColumnRenamed("product_id", "_valid_pid"),
    order_items_typed.product_id == F.col("_valid_pid"),
    "left"
).withColumn("has_valid_product_ref", F.col("_valid_pid").isNotNull())

silver_order_items = (
    order_items_checked.filter(F.col("has_valid_product_ref") == True)
    .select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_bronze_ingested_at")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
)

quarantine_order_items = (
    order_items_checked.filter(F.col("has_valid_product_ref") == False)
    .select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_bronze_ingested_at")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
    .withColumn("quarantine_reason", F.lit("product_id not found in silver_products"))
)

silver_order_items.write.format("delta").mode("overwrite").saveAsTable("silver_order_items")
quarantine_order_items.write.format("delta").mode("overwrite").saveAsTable("silver_quarantine_order_items")

log_dq(
    "silver_order_items",
    total_in=order_items_bronze.count(),
    total_out=silver_order_items.count(),
    quarantined=quarantine_order_items.count(),
    notes="referential integrity check against silver_products",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 8, Finished, Available, Finished, False)

[DQ] silver_order_items: in=6027 out=5896 quarantined=131 (referential integrity check against silver_products)


**Silver: Loyalty**

This is the most involved transformation in Silver, because the loyalty
CSV simulates a legacy system export with three real problems layered
together:

1. **Inconsistent date formats** — `enrollment_date` appears in three
   different formats across records (`MM/dd/yyyy`, `dd-MM-yyyy`,
   `yyyy.MM.dd`). We try each format in order using `coalesce()`; if none
   match, the date stays null and the row is flagged (not rejected) via
   `has_unparsed_date`.
2. **Duplicate rows** — a handful of exact-repeat records (simulating a
   re-run export bug) are removed via `dropDuplicates()`.
3. **Negative points balances** — a known legacy bug produced impossible
   negative point values. These are quarantined, since a negative loyalty
   balance is a business-rule violation, not just a formatting issue.

In [7]:
# ============================================================
# SILVER: LOYALTY
# ============================================================
#
# Business rules:
# 1. Parse enrollment_date using three supported formats.
# 2. Keep unparseable dates but flag them.
# 3. Remove exact duplicate records.
# 4. Quarantine negative points balances.
# 5. Promote referral_code from Bronze to Silver.
# ============================================================


# ------------------------------------------------------------
# 1. Read Bronze loyalty
# ------------------------------------------------------------

loyalty_bronze = spark.table("bronze_loyalty")

print("Bronze loyalty schema:")
loyalty_bronze.printSchema()


# ------------------------------------------------------------
# 2. Parse enrollment dates
# ------------------------------------------------------------

loyalty_dates_parsed = loyalty_bronze.withColumn(
    "enrollment_date_parsed",
    F.coalesce(
        F.to_date(
            F.col("enrollment_date"),
            "MM/dd/yyyy"
        ),
        F.to_date(
            F.col("enrollment_date"),
            "dd-MM-yyyy"
        ),
        F.to_date(
            F.col("enrollment_date"),
            "yyyy.MM.dd"
        ),
    )
)


# ------------------------------------------------------------
# 3. Type conversion and email cleaning
# ------------------------------------------------------------

loyalty_typed = (
    loyalty_dates_parsed
    .withColumn(
        "points_balance",
        F.col("points_balance").cast(IntegerType())
    )
    .withColumn(
        "email_clean",
        F.when(
            (F.trim(F.col("email")) == "") |
            F.col("email").isNull(),
            None
        )
        .otherwise(
            F.lower(F.trim(F.col("email")))
        )
    )
)


# ------------------------------------------------------------
# 4. Remove exact duplicate records
# ------------------------------------------------------------

dedup_cols = [
    "member_id",
    "full_name",
    "email_clean",
    "points_balance",
    "tier",
    "enrollment_date_parsed"
]

loyalty_deduped = loyalty_typed.dropDuplicates(
    dedup_cols
)


# ------------------------------------------------------------
# 5. Separate valid and quarantined records
# ------------------------------------------------------------

loyalty_valid = loyalty_deduped.filter(
    F.col("points_balance") >= 0
)

loyalty_quarantine = loyalty_deduped.filter(
    F.col("points_balance") < 0
)


# ============================================================
# 6. SILVER LOYALTY
# ============================================================

silver_loyalty = (
    loyalty_valid
    .select(
        "member_id",
        "full_name",
        F.col("email_clean").alias("email"),
        "points_balance",
        "tier",
        F.col("enrollment_date_parsed").alias("enrollment_date"),
        "_bronze_ingested_at",
        "_source_system",
        "_source_file",
        "referral_code",
    )
    .withColumn(
        "_silver_processed_at",
        F.lit(SILVER_RUN_TS).cast(TimestampType())
    )
    .withColumn(
        "has_unparsed_date",
        F.col("enrollment_date").isNull()
    )
)

print("\nSilver loyalty DataFrame schema:")
silver_loyalty.printSchema()


# ============================================================
# 7. SILVER LOYALTY QUARANTINE
# ============================================================

quarantine_loyalty_final = (
    loyalty_quarantine
    .select(
        "member_id",
        "full_name",
        F.col("email_clean").alias("email"),
        "points_balance",
        "tier",
        F.col("enrollment_date_parsed").alias("enrollment_date"),
        "_bronze_ingested_at",
        "_source_system",
        "_source_file",
        "referral_code",
    )
    .withColumn(
        "_silver_processed_at",
        F.lit(SILVER_RUN_TS).cast(TimestampType())
    )
    .withColumn(
        "quarantine_reason",
        F.lit("negative points_balance")
    )
)

print("\nSilver loyalty quarantine DataFrame schema:")
quarantine_loyalty_final.printSchema()


# ============================================================
# 8. WRITE SILVER LOYALTY TABLE
# ============================================================
# Uses overwriteSchema instead of DROP TABLE + recreate, so that
# Delta's version history (time travel, per ADR-008) is preserved
# across schema changes rather than destroyed on every run.

silver_loyalty.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_loyalty")

print("\nSilver loyalty table written successfully.")


# ============================================================
# 9. WRITE SILVER LOYALTY QUARANTINE TABLE
# ============================================================

quarantine_loyalty_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_quarantine_loyalty")

print("Silver loyalty quarantine table written successfully.")


# ============================================================
# 10. DATA QUALITY LOGGING
# ============================================================

total_in = loyalty_bronze.count()
total_out = silver_loyalty.count()
total_quarantined = quarantine_loyalty_final.count()

log_dq(
    "silver_loyalty",
    total_in=total_in,
    total_out=total_out,
    quarantined=total_quarantined,
    notes=(
        "3-format date parsing, exact duplicate removal, "
        "negative points quarantined, unparseable dates kept "
        "and flagged, referral_code promoted through schema evolution"
    ),
)


# ============================================================
# 11. UNPARSED DATE CHECK
# ============================================================

unparsed_count = silver_loyalty.filter(
    F.col("has_unparsed_date") == True
).count()

print(
    f"\n[DQ] silver_loyalty: "
    f"{unparsed_count} rows have an unparseable "
    f"enrollment_date (kept, flagged)"
)


# ============================================================
# 12. FINAL SCHEMA VERIFICATION
# ============================================================

print("\nFinal silver_loyalty schema:")
spark.table("silver_loyalty").printSchema()

print("\nFinal silver_quarantine_loyalty schema:")
spark.table("silver_quarantine_loyalty").printSchema()


# ============================================================
# 13. REFERRAL CODE VERIFICATION
# ============================================================

print("\nSilver loyalty referral_code sample:")

spark.table("silver_loyalty").select(
    "member_id",
    "referral_code"
).show(
    5,
    truncate=False
)


# ============================================================
# 14. FINAL DQ SUMMARY
# ============================================================

print(
    f"\n[DQ] silver_loyalty: "
    f"in={total_in} "
    f"out={total_out} "
    f"quarantined={total_quarantined}"
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 9, Finished, Available, Finished, False)

Bronze loyalty schema:
root
 |-- member_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- points_balance: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- enrollment_date: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- referral_code: string (nullable = true)


Silver loyalty DataFrame schema:
root
 |-- member_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- points_balance: integer (nullable = true)
 |-- tier: string (nullable = true)
 |-- enrollment_date: date (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- referral_code: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)
 |--

**Silver: Reviews**

Flattens the nested `review` struct from Bronze (including the
doubly-nested `reviewer.location`) into flat top-level columns — this is
the payoff of using `explode()` in Bronze instead of trying to flatten
everything in one step.

Also performs a referential integrity check against `silver_products`,
since ~2% of reviews were seeded with a stale `product_id` that no longer
exists (simulating a reviews microservice that's out of sync with the
product catalog). Missing ratings are kept and flagged via
`has_missing_rating`, since a review without a star rating is still a
valid, usable review.

In [8]:
reviews_bronze = spark.table("bronze_reviews")
valid_product_ids_r = spark.table("silver_products").select(F.col("product_id").alias("_valid_pid_r"))

reviews_flat = reviews_bronze.select(
    F.col("review.review_id").alias("review_id"),
    F.col("review.product_id").cast(IntegerType()).alias("product_id"),
    F.col("review.reviewer.name").alias("reviewer_name"),
    F.col("review.reviewer.verified_purchase").alias("verified_purchase"),
    F.col("review.reviewer.location.city").alias("reviewer_city"),
    F.col("review.reviewer.location.country").alias("reviewer_country"),
    F.col("review.rating").cast(IntegerType()).alias("rating"),
    F.col("review.review_text").alias("review_text"),
    F.to_timestamp(F.col("review.review_date")).alias("review_date"),
    F.col("review.helpful_votes").cast(IntegerType()).alias("helpful_votes"),
    F.col("review.tags").alias("tags"),
    F.col("_bronze_ingested_at"),
)

reviews_checked = reviews_flat.join(
    valid_product_ids_r,
    reviews_flat.product_id == F.col("_valid_pid_r"),
    "left"
).withColumn("has_valid_product_ref", F.col("_valid_pid_r").isNotNull())

silver_reviews = (
    reviews_checked.filter(F.col("has_valid_product_ref") == True)
    .drop("_valid_pid_r", "has_valid_product_ref")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
    .withColumn("has_missing_rating", F.col("rating").isNull())
)

quarantine_reviews = (
    reviews_checked.filter(F.col("has_valid_product_ref") == False)
    .drop("_valid_pid_r", "has_valid_product_ref")
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
    .withColumn("quarantine_reason", F.lit("product_id not found in silver_products"))
)

silver_reviews.write.format("delta").mode("overwrite").saveAsTable("silver_reviews")
quarantine_reviews.write.format("delta").mode("overwrite").saveAsTable("silver_quarantine_reviews")

log_dq(
    "silver_reviews",
    total_in=reviews_bronze.count(),
    total_out=silver_reviews.count(),
    quarantined=quarantine_reviews.count(),
    notes="struct flattened, referential check against silver_products; null ratings kept+flagged",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 10, Finished, Available, Finished, False)

[DQ] silver_reviews: in=400 out=394 quarantined=6 (struct flattened, referential check against silver_products; null ratings kept+flagged)


**Silver: Exchange Rates**

Explodes the `raw_response.rates` map (one key-value pair per currency)
into individual rows, so it can be joined cleanly against orders by
currency code in the Gold layer. No quality issues seeded here — this is
authoritative external data from a trusted API.

In [9]:
from pyspark.sql.types import MapType, StringType

rates_bronze = spark.table("bronze_exchange_rates")

# raw_response.rates was inferred as a STRUCT (fixed named fields), not a MAP,
# because JSON schema inference treats objects with named keys as structs.
# We force it into a proper MAP<STRING, DOUBLE> by round-tripping through JSON text,
# which also resolves the IDR field's inconsistent BIGINT-vs-DOUBLE inference.
rates_as_map = rates_bronze.withColumn(
    "rates_json", F.to_json(F.col("raw_response.rates"))
).withColumn(
    "rates_map", F.from_json(F.col("rates_json"), MapType(StringType(), DoubleType()))
)

silver_exchange_rates = (
    rates_as_map
    .select(
        F.col("raw_response.base").alias("base_currency"),
        F.to_date(F.col("raw_response.date")).alias("rate_date"),
        F.explode(F.col("rates_map")).alias("currency_code", "exchange_rate"),
        "_bronze_ingested_at",
    )
    .withColumn("_silver_processed_at", F.lit(SILVER_RUN_TS).cast(TimestampType()))
)

silver_exchange_rates.write.format("delta").mode("overwrite").saveAsTable("silver_exchange_rates")

log_dq(
    "silver_exchange_rates",
    total_in=rates_bronze.count(),
    total_out=silver_exchange_rates.count(),
    quarantined=0,
    notes="struct-to-map conversion via JSON round-trip, then exploded to one row per currency",
)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 11, Finished, Available, Finished, False)

[DQ] silver_exchange_rates: in=1 out=29 quarantined=0 (struct-to-map conversion via JSON round-trip, then exploded to one row per currency)


**Silver Layer: DQ Log & Summary**

Writes the accumulated data quality results from every table processed in
this run into a persistent `silver_dq_log` Delta table (appended, so
history across runs is preserved — not overwritten). This is the
foundation for pipeline observability in a later phase: without this log,
questions like "did quarantine rates spike this week?" would be
unanswerable after the fact.

In [10]:
dq_log_df = spark.createDataFrame(dq_log_rows)
dq_log_df.write.format("delta").mode("append").saveAsTable("silver_dq_log")

print("=" * 60)
print("SILVER LAYER RUN SUMMARY")
print("=" * 60)
dq_log_df.select("table_name", "rows_in", "rows_out", "rows_quarantined", "quarantine_rate_pct").show(truncate=False)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 12, Finished, Available, Finished, False)

SILVER LAYER RUN SUMMARY
+---------------------+-------+--------+----------------+-------------------+
|table_name           |rows_in|rows_out|rows_quarantined|quarantine_rate_pct|
+---------------------+-------+--------+----------------+-------------------+
|silver_customers     |520    |510     |10              |1.92               |
|silver_products      |100    |100     |0               |0.0                |
|silver_orders        |2003   |2003    |0               |0.0                |
|silver_order_items   |6027   |5896    |131             |2.17               |
|silver_loyalty       |305    |289     |11              |3.61               |
|silver_reviews       |400    |394     |6               |1.5                |
|silver_exchange_rates|1      |29      |0               |0.0                |
+---------------------+-------+--------+----------------+-------------------+



In [11]:
spark.table("silver_customers").count()

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 13, Finished, Available, Finished, False)

510

In [12]:
spark.table("silver_order_items").filter(F.col("order_id").isin([2001, 2002, 2003])).show()

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 14, Finished, Available, Finished, False)

+-------------+--------+----------+--------+----------+--------------------+--------------------+
|order_item_id|order_id|product_id|quantity|unit_price| _bronze_ingested_at|_silver_processed_at|
+-------------+--------+----------+--------+----------+--------------------+--------------------+
|         6025|    2001|        15|       3|    482.64|2026-09-02 19:54:...|2026-09-03 07:48:...|
|         6026|    2002|        89|       1|    145.48|2026-09-02 19:54:...|2026-09-03 07:48:...|
|         6027|    2003|        86|       2|    396.18|2026-09-02 19:54:...|2026-09-03 07:48:...|
+-------------+--------+----------+--------+----------+--------------------+--------------------+



In [13]:
spark.table("silver_dq_log").count()

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 15, Finished, Available, Finished, False)

48

In [14]:
spark.table("silver_dq_log").groupBy("run_ts").count().orderBy("run_ts").show(truncate=False)

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 16, Finished, Available, Finished, False)

+--------------------------------+-----+
|run_ts                          |count|
+--------------------------------+-----+
|2026-08-27T04:05:13.257628      |7    |
|2026-08-27T06:41:30.344325      |7    |
|2026-08-27T20:39:26.140624      |7    |
|2026-08-28T07:47:42.867099      |7    |
|2026-09-02T19:18:25.858401+00:00|6    |
|2026-09-02T19:50:51.128472+00:00|7    |
|2026-09-03T07:48:41.254465+00:00|7    |
+--------------------------------+-----+



In [15]:
spark.table("silver_customers").filter(F.col("customer_id").isin([9001, 9002, 9003])).show()

StatementMeta(, 8aa0720d-d4d2-4ad6-b43f-dbacda5d5ed4, 17, Finished, Available, Finished, False)

+-----------+----------+---------+-----+-----+----+-------+-------------------+--------------------+-----------------+
|customer_id|first_name|last_name|email|phone|city|country|_bronze_ingested_at|_silver_processed_at|has_missing_email|
+-----------+----------+---------+-----+-----+----+-------+-------------------+--------------------+-----------------+
+-----------+----------+---------+-----+-----+----+-------+-------------------+--------------------+-----------------+

